In [1]:
# =============================================================================
# ShiftGuard10 — Final Solution  |  Single Fold, 200 Epochs, T4 x2
# =============================================================================
# Philosophy: keep everything that PROVED to work at 0.831, only add
# things that are guaranteed safe improvements.
#
# Kept from original (proven):
#   ✓ WideResNet-28-10
#   ✓ RandAugment(n=2, m=9)  ← NOT replaced with AugMix
#   ✓ LR = 0.1               ← NOT scaled with batch size
#   ✓ CosineAnnealingWarmRestarts(T0=50)
#   ✓ Mixup + CutMix + RandomErasing
#   ✓ WeightedRandomSampler
#   ✓ TTA × 10
#
# Safe additions only:
#   + 200 epochs (was 100) with SWA over last 40
#   + DropPath=0.1 inside WideResNet blocks
#   + DataParallel across both T4s
#   + Proper 5% val split (was val=train which was a bug)
#   + All FutureWarnings fixed (torch.amp)
#   + Mixed precision (AMP) for speed
#
# Expected: 0.88–0.91  (from 0.831 baseline)
# Time:     ~7.5 hrs on T4 x2  (fits 12hr session)
# =============================================================================

import os, random, time, math
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast          # ✓ no FutureWarning
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing  # ← proven, kept

# ── 0. GPU Setup ──────────────────────────────────────────────────────────────
print(f"GPUs available : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS  = torch.cuda.device_count()
USE_MULTI = NUM_GPUS > 1
print(f"Mode: {'DataParallel — both T4s active' if USE_MULTI else 'Single GPU'}\n")

# ── 1. Config ─────────────────────────────────────────────────────────────────
class CFG:
    seed         = 42
    num_classes  = 10
    batch_size   = 256 if USE_MULTI else 128   # double batch for dual GPU
    num_workers  = 4
    epochs       = 200
    swa_start    = 160
    swa_lr       = 0.005                       # conservative SWA LR

    # ✓ LR stays at 0.1 — linear scaling rule does NOT help at this batch size
    lr           = 0.1
    momentum     = 0.9
    weight_decay = 5e-4
    nesterov     = True

    # Cosine warm restarts — same as original working code
    T0           = 50
    T_mult       = 1
    eta_min      = 1e-6

    # 5% val split — small enough to keep training data, real enough to signal
    val_frac     = 0.05

    # RandAugment — proven to work, NOT replaced
    randaug_n    = 2
    randaug_m    = 9

    # Mixup / CutMix
    mixup_alpha  = 0.4
    cutmix_alpha = 1.0
    mix_prob     = 0.5
    label_smooth = 0.1

    # TTA
    tta_n        = 10

    # Paths
    base_dir     = Path('/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge')
    train_dir    = base_dir / 'train_images'
    test_dir     = base_dir / 'test_images'
    train_csv    = base_dir / 'train_labels.csv'
    sample_sub   = base_dir / 'sample_submission.csv'
    out_dir      = Path('/kaggle/working')

est = CFG.epochs * (130 if USE_MULTI else 240) / 60
print(f"Batch size : {CFG.batch_size}")
print(f"LR         : {CFG.lr}  (fixed — not scaled)")
print(f"Epochs     : {CFG.epochs}  |  SWA from epoch {CFG.swa_start}")
print(f"Val split  : {int(CFG.val_frac*100)}%  (~{int(29400*CFG.val_frac):,} held out)")
print(f"Est. time  : {est:.0f} min  ({est/60:.1f} hrs)\n")

# ── 2. Reproducibility ────────────────────────────────────────────────────────
def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(CFG.seed)

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2CLS = {i: c for c, i in CLS2IDX.items()}
MEAN    = [0.4914, 0.4822, 0.4465]
STD     = [0.2470, 0.2435, 0.2616]

# ── 3. Transforms (RandAugment kept — proven) ─────────────────────────────────
def build_train_transforms():
    return transforms.Compose([
        transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
        transforms.RandomHorizontalFlip(),
        RandAugment(num_ops=CFG.randaug_n, magnitude=CFG.randaug_m),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
        RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random'),
    ])

def build_val_transforms():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

def build_tta_transforms():
    tfms    = []
    offsets = [(0,0),(0,8),(8,0),(8,8),(4,4)]
    for flip in [False, True]:
        for r, c in offsets:
            ops = []
            if flip:
                ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops.append(transforms.Pad(4, padding_mode='reflect'))
            ops.append(transforms.Lambda(
                lambda img, r=r, c=c: transforms.functional.crop(img, r, c, 32, 32)))
            ops += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
            tfms.append(transforms.Compose(ops))
    return tfms[:CFG.tta_n]

TTA_TRANSFORMS = build_tta_transforms()

# ── 4. Dataset ────────────────────────────────────────────────────────────────
class ImageDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, labelled=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.labelled  = labelled
        # pre-resolve paths once at init
        self.paths = []
        for _, row in self.df.iterrows():
            img_id = str(row['id']).zfill(6)
            p      = img_dir / f'{img_id}.png'
            if not p.exists():
                p = img_dir / f"{row['id']}.png"
            self.paths.append(p)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.labelled:
            return img, int(self.df.iloc[idx]['label_idx'])
        return img, str(self.df.iloc[idx]['id'])

# ── 5. WideResNet-28-10 + DropPath (new, safe addition) ──────────────────────
class DropPath(nn.Module):
    """Stochastic depth — drops entire residual branch randomly in training."""
    def __init__(self, p=0.1):
        super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        keep = 1 - self.p
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep
        return x * mask.float() / keep

class _WideBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        self.bn1  = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2  = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.drop = drop
        self.dp   = DropPath(drop_path)
        self.skip = None if (in_ch == out_ch and stride == 1) else \
                    nn.Conv2d(in_ch, out_ch, 1, stride, 0, bias=False)

    def forward(self, x):
        h   = F.relu(self.bn1(x), inplace=True)
        out = self.conv1(h)
        out = F.relu(self.bn2(out), inplace=True)
        if self.drop > 0 and self.training:
            out = F.dropout(out, p=self.drop)
        out = self.dp(self.conv2(out))
        return out + (self.skip(h) if self.skip else x)

class _WideGroup(nn.Module):
    def __init__(self, n, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        layers = [_WideBlock(in_ch, out_ch, stride, drop, drop_path)]
        for _ in range(n - 1):
            layers.append(_WideBlock(out_ch, out_ch, 1, drop, drop_path))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class WideResNet(nn.Module):
    def __init__(self, depth=28, width=10, num_classes=10, drop=0.3, drop_path=0.1):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n   = (depth - 4) // 6
        chs = [16, 16*width, 32*width, 64*width]
        self.conv0  = nn.Conv2d(3, chs[0], 3, 1, 1, bias=False)
        self.group1 = _WideGroup(n, chs[0], chs[1], 1, drop, drop_path)
        self.group2 = _WideGroup(n, chs[1], chs[2], 2, drop, drop_path)
        self.group3 = _WideGroup(n, chs[2], chs[3], 2, drop, drop_path)
        self.bn     = nn.BatchNorm2d(chs[3])
        self.fc     = nn.Linear(chs[3], num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.conv0(x)
        x = self.group1(x); x = self.group2(x); x = self.group3(x)
        x = F.relu(self.bn(x), inplace=True)
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def build_model():
    m = WideResNet(28, 10, CFG.num_classes, drop=0.3, drop_path=0.1).to(DEVICE)
    if USE_MULTI:
        m = nn.DataParallel(m)
        print(f"  DataParallel across {NUM_GPUS} GPUs")
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  Params: {n/1e6:.2f}M")
    return m

# ── 6. Loss ───────────────────────────────────────────────────────────────────
class SoftCE(nn.Module):
    def __init__(self, eps=0.1):
        super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        C = logits.size(1)
        if targets.dim() == 1:
            targets = F.one_hot(targets, C).float()
        targets = (1 - self.eps) * targets + self.eps / C
        return -(targets * F.log_softmax(logits, 1)).sum(1).mean()

criterion = SoftCE(CFG.label_smooth)

# ── 7. Mixup / CutMix ────────────────────────────────────────────────────────
def rand_bbox(size, lam):
    H, W = size[-2], size[-1]
    r    = math.sqrt(1 - lam)
    h, w = int(H*r), int(W*r)
    cy, cx = random.randint(0, H), random.randint(0, W)
    return max(0,cx-w//2), max(0,cy-h//2), min(W,cx+w//2), min(H,cy+h//2)

def apply_mix(imgs, lbls):
    perm = torch.randperm(imgs.size(0), device=imgs.device)
    oh   = F.one_hot(lbls, CFG.num_classes).float()
    if random.random() < 0.5:
        lam  = np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha)
        imgs = lam * imgs + (1-lam) * imgs[perm]
    else:
        lam = np.random.beta(CFG.cutmix_alpha, CFG.cutmix_alpha)
        x1,y1,x2,y2 = rand_bbox(imgs.shape, lam)
        imgs = imgs.clone()
        imgs[:,:,y1:y2,x1:x2] = imgs[perm,:,y1:y2,x1:x2]
        lam = 1 - (x2-x1)*(y2-y1)/(imgs.shape[-1]*imgs.shape[-2])
    return imgs, lam*oh + (1-lam)*oh[perm]

# ── 8. Weighted Sampler ───────────────────────────────────────────────────────
def make_sampler(label_list):
    counts  = np.bincount(label_list, minlength=CFG.num_classes).astype(float)
    weights = 1.0 / (counts + 1e-6)
    sw      = torch.tensor([weights[l] for l in label_list], dtype=torch.float)
    return WeightedRandomSampler(sw, len(sw), replacement=True)

# ── 9. Train one epoch ────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    running = 0.0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        if random.random() < CFG.mix_prob:
            imgs, soft = apply_mix(imgs, lbls)
        else:
            soft = lbls
        optimizer.zero_grad()
        with autocast('cuda'):
            loss = criterion(model(imgs), soft)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        params = model.module.parameters() if USE_MULTI else model.parameters()
        nn.utils.clip_grad_norm_(params, 5.0)
        scaler.step(optimizer); scaler.update()
        running += loss.item()
    return running / len(loader)

# ── 10. Validate ──────────────────────────────────────────────────────────────
@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds, trues = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE)
        with autocast('cuda'):
            p = model(imgs).argmax(1).cpu().numpy()
        preds.extend(p); trues.extend(lbls.numpy())
    return (f1_score(trues, preds, average='macro'),
            np.mean(np.array(preds) == np.array(trues)))

# ── 11. TTA Inference ─────────────────────────────────────────────────────────
@torch.no_grad()
def predict_tta(model, df, img_dir):
    model.eval()
    all_probs = []
    for i, tf in enumerate(TTA_TRANSFORMS, 1):
        ds = ImageDataset(df, img_dir, transform=tf, labelled=False)
        ld = DataLoader(ds, CFG.batch_size*2, num_workers=CFG.num_workers,
                        shuffle=False, pin_memory=True)
        probs = []
        for imgs, _ in ld:
            with autocast('cuda'):
                logits = model(imgs.to(DEVICE))
            probs.append(F.softmax(logits, 1).cpu().float().numpy())
        all_probs.append(np.concatenate(probs, 0))
        print(f"    TTA {i}/{CFG.tta_n}", end='\r')
    print()
    return np.mean(all_probs, 0)

# ── 12. Load & Split Data ─────────────────────────────────────────────────────
print('Loading data …')
train_df = pd.read_csv(CFG.train_csv)
train_df.columns = [c.strip().lower() for c in train_df.columns]
if 'label' in train_df.columns:
    train_df['label_idx'] = train_df['label'].map(CLS2IDX)
elif 'class' in train_df.columns:
    train_df['label_idx'] = train_df['class'].map(CLS2IDX)

test_df = pd.read_csv(CFG.sample_sub)
test_df.columns = [c.strip().lower() for c in test_df.columns]

# 5% real validation split — small but honest
tr_df, va_df = train_test_split(
    train_df, test_size=CFG.val_frac,
    stratify=train_df['label_idx'], random_state=CFG.seed
)
tr_df = tr_df.reset_index(drop=True)
va_df = va_df.reset_index(drop=True)
print(f"Train : {len(tr_df):,}  |  Val : {len(va_df):,}  |  Test : {len(test_df):,}")

# ── 13. DataLoaders ───────────────────────────────────────────────────────────
tr_ds   = ImageDataset(tr_df, CFG.train_dir, build_train_transforms(), labelled=True)
va_ds   = ImageDataset(va_df, CFG.train_dir, build_val_transforms(),   labelled=True)

sampler = make_sampler(tr_df['label_idx'].tolist())
tr_ld   = DataLoader(tr_ds, CFG.batch_size, sampler=sampler,
                     num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
va_ld   = DataLoader(va_ds, CFG.batch_size*2, shuffle=False,
                     num_workers=CFG.num_workers, pin_memory=True)

# ── 14. Model / Optimizer / Scheduler ────────────────────────────────────────
print('\nBuilding model …')
model     = build_model()
raw_model = model.module if USE_MULTI else model
raw_params = raw_model.parameters()

optimizer = optim.SGD(raw_params, lr=CFG.lr,
                      momentum=CFG.momentum, weight_decay=CFG.weight_decay,
                      nesterov=CFG.nesterov)

# Same scheduler as original working code — proven
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG.T0, T_mult=CFG.T_mult, eta_min=CFG.eta_min
)

scaler    = GradScaler('cuda')

swa_model = AveragedModel(raw_model)
swa_sched = SWALR(optimizer, swa_lr=CFG.swa_lr, anneal_epochs=10)

# ── 15. Training Loop ─────────────────────────────────────────────────────────
best_f1   = 0.0
best_ckpt = CFG.out_dir / 'best_model.pth'
log_rows  = []
t_total   = time.time()

print(f"\n{'Epoch':>6} {'Loss':>8} {'Val F1':>8} {'Val Acc':>9} {'LR':>10} {'Time':>7}")
print('─' * 58)

for epoch in range(1, CFG.epochs + 1):
    t0 = time.time()

    tr_loss = train_one_epoch(model, tr_ld, optimizer, scaler)

    # scheduler: warm restarts until SWA kicks in
    if epoch < CFG.swa_start:
        scheduler.step(epoch)
    else:
        swa_model.update_parameters(raw_model)
        swa_sched.step()

    val_f1, val_acc = validate(model, va_ld)
    lr_now = optimizer.param_groups[0]['lr']

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(raw_model.state_dict(), best_ckpt)

    log_rows.append({'epoch': epoch, 'loss': tr_loss,
                     'val_f1': val_f1, 'val_acc': val_acc, 'lr': lr_now})

    if epoch % 10 == 0 or epoch <= 5:
        star = ' ★' if val_f1 >= best_f1 else ''
        dt   = time.time() - t0
        print(f'{epoch:>6} {tr_loss:>8.4f} {val_f1:>8.4f} '
              f'{val_acc*100:>8.2f}% {lr_now:>10.6f} {dt:>5.0f}s{star}')

elapsed = (time.time() - t_total) / 3600
print(f'\nTraining done in {elapsed:.2f} hrs  |  Best val F1 : {best_f1:.4f}')

# ── 16. SWA BN Update + Model Selection ──────────────────────────────────────
print('\nUpdating SWA batch-norm statistics …')
update_bn(tr_ld, swa_model, device=DEVICE)
swa_f1, swa_acc = validate(swa_model, va_ld)
print(f'SWA model  → Val F1 : {swa_f1:.4f}  |  Acc : {swa_acc*100:.2f}%')
print(f'Best epoch → Val F1 : {best_f1:.4f}')

if swa_f1 >= best_f1:
    infer_model = swa_model
    print('Using SWA model for inference.')
else:
    raw_model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    infer_model = raw_model
    print(f'Using best checkpoint (F1={best_f1:.4f}) for inference.')

pd.DataFrame(log_rows).to_csv(CFG.out_dir / 'training_log.csv', index=False)

# ── 17. TTA Inference + Submission ───────────────────────────────────────────
print(f'\nRunning TTA inference ({CFG.tta_n} passes) …')
probs       = predict_tta(infer_model, test_df, CFG.test_dir)
pred_labels = [IDX2CLS[i] for i in probs.argmax(1)]

sub = pd.DataFrame({'id': test_df['id'], 'label': pred_labels})
sub.to_csv(CFG.out_dir / 'submission.csv', index=False)
print(f'\n✅  submission.csv saved  ({len(sub):,} rows)')
print(sub.head(10).to_string(index=False))
print('\nPredicted class distribution:')
print(sub['label'].value_counts().to_string())

GPUs available : 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Mode: DataParallel — both T4s active

Batch size : 256
LR         : 0.1  (fixed — not scaled)
Epochs     : 200  |  SWA from epoch 160
Val split  : 5%  (~1,470 held out)
Est. time  : 433 min  (7.2 hrs)

Loading data …
Train : 27,930  |  Val : 1,470  |  Test : 7,600

Building model …
  DataParallel across 2 GPUs
  Params: 36.48M

 Epoch     Loss   Val F1   Val Acc         LR    Time
──────────────────────────────────────────────────────────
     1   2.2022   0.1825    20.14%   0.099901    50s ★
     2   2.0417   0.2295    32.24%   0.099606    45s ★
     3   1.9649   0.2913    34.15%   0.099114    48s ★
     4   1.8594   0.3774    46.19%   0.098429    52s ★
     5   1.8102   0.3925    47.82%   0.097553    50s ★
    10   1.6129   0.4967    56.60%   0.090451    51s ★
    20   1.3886   0.6343    71.36%   0.065451    50s
    30   1.2430   0.7154    77.62%   0.034550    50s ★
    40   1.0724   0.7569    84.90%   0.009550    50s
    50   1.